# Probability of Default (PD) Model
**RAKBANK Management Associate Programme - Credit Risk Case Study 1**
**Author:** Theo Li | **Date:** June 2026

---

### Objective
Build a leakage-free, interpretable PD model that predicts the probability a loan applicant will default, and translate scores into an Approve / Review / Decline lending policy.

### Structure
1. Setup and Imports
2. Load Data
3. Exploratory Data Analysis
4. Feature Engineering
5. Train / Validation Split
6. Preprocessing Pipeline
7. Model Training
8. Evaluation
9. SHAP Explainability
10. Risk Banding


## 1. Setup and Imports

In [ ]:
# Uncomment to install if needed:
# !pip install pandas numpy scikit-learn shap matplotlib seaborn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
)

sns.set_theme(style="whitegrid", font_scale=1.05)
BLUE, RED, AMBER, GREEN = "#2563EB", "#DC2626", "#F59E0B", "#16A34A"
print("Libraries loaded.")

## 2. Load Data

In [ ]:
df = pd.read_csv("data/case1_credit_approval.csv.gz")
print(f"Shape        : {df.shape}")
print(f"Default rate : {df['TARGET'].mean():.1%}")
df.head(3)

## 3. Exploratory Data Analysis

Before building anything, three questions need answering:
1. How imbalanced is the target variable?
2. Do EXT_SOURCE scores separate defaulters from non-defaulters?
3. What does missing bureau data imply about those applicants?


### 3.1 Class Imbalance

In [ ]:
vc = df["TARGET"].value_counts()
print(f"No Default : {vc[0]:,} ({vc[0]/len(df):.1%})")
print(f"Default    : {vc[1]:,} ({vc[1]/len(df):.1%})")
print()
print("With only 8.1% defaulters, a model predicting 'no default' for everyone")
print("scores 91.9% accuracy while catching zero actual defaulters.")
print("This is why we use AUC-ROC, Gini, and KS - not accuracy.")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["No Default (0)", "Default (1)"], vc.values,
       color=[BLUE, RED], edgecolor="white", width=0.5)
ax.set_title("Target Class Distribution", fontweight="bold")
ax.set_ylabel("Count")
for i, v in enumerate(vc.values):
    ax.text(i, v + 200, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

### 3.2 Missing Values

In [ ]:
miss = df.isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print("Columns with missing values (%):")
print((miss * 100).round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(miss.index, miss.values * 100, color=AMBER, edgecolor="white")
ax.set_title("Missing Values by Column (%)", fontweight="bold")
ax.set_xlabel("% Missing")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3.3 EXT_SOURCE Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]):
    for tgt, color, label in [(0, BLUE, "No Default"), (1, RED, "Default")]:
        vals = df.loc[df["TARGET"] == tgt, col].dropna()
        axes[i].hist(vals, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(col, fontweight="bold")
    axes[i].legend()
    axes[i].set_xlabel("Score")
    axes[i].set_ylabel("Density")
plt.suptitle("EXT_SOURCE Scores - Defaulters Cluster at Lower Scores",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Finding: Clear separation between defaulters and non-defaulters.")
print("EXT_SOURCE will be the most predictive features in the model.")

### 3.4 Bureau File Missingness

In [ ]:
no_bureau  = df[df["HAS_BUREAU_FILE"] == 0]
has_bureau = df[df["HAS_BUREAU_FILE"] == 1]
dr_no  = no_bureau["TARGET"].mean()
dr_has = has_bureau["TARGET"].mean()

print(f"No bureau file : {len(no_bureau):,} applicants ({len(no_bureau)/len(df):.1%}) | Default rate: {dr_no:.1%}")
print(f"Has bureau file: {len(has_bureau):,} applicants ({len(has_bureau)/len(df):.1%}) | Default rate: {dr_has:.1%}")
print()
print("Interpretation: The same 8,752 applicants are missing EVERY bureau column.")
print("This means they have never borrowed before - not that their data is corrupted.")
print("These thin-file applicants have a LOWER default rate (3.9% vs 9.0%).")
print("Auto-declining them would lose 17.5% of applicants, many of whom are creditworthy.")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["No Bureau File", "Has Bureau File"],
              [dr_no * 100, dr_has * 100],
              color=[AMBER, BLUE], width=0.4, edgecolor="white")
ax.set_ylabel("Default Rate (%)")
ax.set_title("Default Rate by Bureau File Presence", fontweight="bold")
for bar, val in zip(bars, [dr_no, dr_has]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{val:.1%}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Raw fields like `DAYS_BIRTH = -15,140` carry limited meaning. We construct 10 features
across three MECE dimensions that cover what a lender needs to know.

| Dimension | Question |
|---|---|
| **Affordability** | Can this person sustain the repayments? |
| **Stability** | How predictable is their financial situation? |
| **Credit History** | What does past behaviour tell us? |

These three dimensions are mutually exclusive and collectively exhaustive. An applicant
could be affordable on paper but have a poor repayment history, or highly stable but
over-leveraged. Together they give a complete picture of credit risk.


In [ ]:
def engineer_features(df):
    d = df.copy()

    # STABILITY
    d["AGE_YEARS"]        = (-d["DAYS_BIRTH"]).clip(lower=0) / 365.25
    d["EMPLOYMENT_YEARS"] = (-d["DAYS_EMPLOYED"]).clip(lower=0) / 365.25

    # AFFORDABILITY
    d["ANNUITY_TO_INCOME"]     = d["AMT_ANNUITY"]       / (d["AMT_INCOME_TOTAL"] + 1)
    d["CREDIT_TO_INCOME"]      = d["AMT_CREDIT"]        / (d["AMT_INCOME_TOTAL"] + 1)
    d["BUREAU_DEBT_TO_INCOME"] = d["BUREAU_TOTAL_DEBT"] / (d["AMT_INCOME_TOTAL"] + 1)
    d["LOAN_COST_RATIO"]       = (d["AMT_ANNUITY"] * d["TERM_MONTHS"]) / (d["AMT_CREDIT"] + 1)

    # CREDIT HISTORY
    dpd_cols = ["BUREAU_DPD_30_COUNT", "BUREAU_DPD_60_COUNT", "BUREAU_DPD_90_COUNT"]
    d["BUREAU_TOTAL_DPD"]        = d[dpd_cols].fillna(0).sum(axis=1)
    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    d["EXT_SOURCE_MEAN"]         = d[ext_cols].mean(axis=1)
    d["EXT_SOURCE_MIN"]          = d[ext_cols].min(axis=1)
    d["BUREAU_UTILIZATION_SAFE"] = d["BUREAU_UTILIZATION"].fillna(0)

    return d

df = engineer_features(df)
print(f"Shape after feature engineering: {df.shape}")

## 5. Stratified Train / Validation Split

The split is stratified on TARGET to preserve the 8.1% default rate in both sets.


In [ ]:
DROP     = ["SK_ID_CURR", "TARGET", "APPLICANT_SCENARIO", "DAYS_BIRTH", "DAYS_EMPLOYED"]
FEATURES = [c for c in df.columns if c not in DROP]
X, y     = df[FEATURES], df["TARGET"]

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(X, y))
X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

print(f"Training set   : {X_train.shape[0]:,} rows | Default rate: {y_train.mean():.1%}")
print(f"Validation set : {X_val.shape[0]:,} rows | Default rate: {y_val.mean():.1%}")

## 6. Preprocessing Pipeline

**Why fit only on training data?**

If we compute the imputation median on all 50,000 rows before splitting, the validation
set's values have already influenced that median. The model has indirectly seen the test
data and performance metrics will be inflated.

`sklearn.Pipeline` enforces the correct order automatically.


In [ ]:
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

# Numeric: fill missing with median (from train only), then standardise
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# Categorical: fill missing with most frequent, then encode as integers
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

print(f"Numeric features     : {len(num_cols)}")
print(f"Categorical features : {len(cat_cols)} -> {cat_cols}")
print("Preprocessor configured.")

## 7. Model Training - Logistic Regression (Baseline)

Logistic Regression is the industry standard baseline for credit scoring.
It assigns a weight to each feature, sums the weighted values, and converts
the result to a probability between 0 and 1.

`class_weight='balanced'` tells the model to weight each defaulter approximately
11 times more than each non-defaulter, preventing it from ignoring the minority class.


In [ ]:
lr_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf",  LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        C=0.1,
        random_state=42,
    )),
])

lr_pipe.fit(X_train, y_train)
lr_prob = lr_pipe.predict_proba(X_val)[:, 1]
print("Model trained.")

# Show top coefficients
clf       = lr_pipe.named_steps["clf"]
prep      = lr_pipe.named_steps["prep"]
all_names = num_cols + cat_cols

coef_df = pd.DataFrame({
    "feature": all_names,
    "coefficient": clf.coef_[0],
}).assign(abs_coef=lambda x: x["coefficient"].abs()
).sort_values("abs_coef", ascending=False).head(12).drop("abs_coef", axis=1)

coef_df["effect"] = coef_df["coefficient"].apply(
    lambda x: "Increases risk" if x > 0 else "Decreases risk"
)
print("\nTop 12 feature coefficients (model weights):")
print(coef_df.to_string(index=False))

## 8. Evaluation

We report four metrics instead of accuracy, because accuracy is misleading
under class imbalance.


In [ ]:
lr_auc  = roc_auc_score(y_val, lr_prob)
lr_gini = 2 * lr_auc - 1
lr_ap   = average_precision_score(y_val, lr_prob)
fpr, tpr, _ = roc_curve(y_val, lr_prob)
lr_ks   = float(np.max(tpr - fpr))

print("Model Performance (Validation Set - 10,000 applicants):")
print(f"  AUC-ROC       : {lr_auc:.4f}  (above 0.8 = strong)")
print(f"  Gini          : {lr_gini:.4f}  (above 0.7 = industry threshold)")
print(f"  KS Statistic  : {lr_ks:.4f}  (good separation)")
print(f"  Avg Precision : {lr_ap:.4f}  (solid minority class performance)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(fpr, tpr, color=BLUE, lw=2, label=f"Logistic Regression (AUC={lr_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC=0.500)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve", fontweight="bold")
axes[0].legend(fontsize=9)

prec, rec, _ = precision_recall_curve(y_val, lr_prob)
axes[1].plot(rec, prec, color=BLUE, lw=2, label=f"AP={lr_ap:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve", fontweight="bold")
axes[1].legend(fontsize=9)

axes[2].hist(lr_prob[y_val == 0], bins=50, alpha=0.6, color=BLUE,
             label="No Default", density=True)
axes[2].hist(lr_prob[y_val == 1], bins=50, alpha=0.6, color=RED,
             label="Default", density=True)
axes[2].set_xlabel("Predicted PD Score")
axes[2].set_ylabel("Density")
axes[2].set_title("PD Score Distribution by Class", fontweight="bold")
axes[2].legend()

plt.suptitle("Model Evaluation - Logistic Regression", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 9. SHAP Explainability

**Coefficients vs SHAP:**
- Coefficients show the model's fixed weights, the same for every applicant
- SHAP breaks down each individual prediction into feature contributions, taking into account that person's actual values

### 9.1 Global Feature Drivers


In [ ]:
X_val_prep    = prep.transform(X_val)
X_val_prep_df = pd.DataFrame(X_val_prep, columns=all_names)

explainer   = shap.LinearExplainer(clf, X_val_prep_df)
shap_values = explainer.shap_values(X_val_prep_df)

print("Reading the plot:")
print("  Each dot = one applicant")
print("  Red = high feature value | Blue = low feature value")
print("  Right = increases default probability | Left = decreases it")

shap.summary_plot(shap_values, X_val_prep_df, max_display=15, show=False)
plt.title("SHAP Summary - Key Drivers of Default Risk", fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

### 9.2 High-Risk Applicant - Individual Explanation

In [ ]:
val_df             = X_val.copy().reset_index(drop=True)
val_df["PD_SCORE"] = lr_prob
val_df["ACTUAL"]   = y_val.values

hr_idx = val_df[val_df["ACTUAL"] == 1]["PD_SCORE"].idxmax()
hr     = val_df.loc[hr_idx]

print(f"Predicted PD     : {hr['PD_SCORE']:.1%}")
print(f"Annuity / Income : {hr['ANNUITY_TO_INCOME']:.1%}")
print(f"Credit / Income  : {hr['CREDIT_TO_INCOME']:.2f}x")
print(f"EXT_SOURCE_MEAN  : {hr['EXT_SOURCE_MEAN']:.3f}")

shap_row    = shap_values[hr_idx]
shap_series = pd.Series(shap_row, index=all_names)
shap_top    = shap_series[shap_series.abs().nlargest(12).index].sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = [RED if v > 0 else BLUE for v in shap_top.values]
ax.barh(range(len(shap_top)), shap_top.values, color=colors, height=0.6, edgecolor="white")
ax.set_yticks(range(len(shap_top)))
ax.set_yticklabels(shap_top.index, fontsize=9)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("SHAP Value (positive = increases default risk)")
ax.set_title(f"High-Risk Applicant Explanation | Predicted PD: {hr['PD_SCORE']:.1%}",
             fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Decision: Decline")
print(f"Repayment burden at {hr['ANNUITY_TO_INCOME']:.1%} of income is the primary driver.")

## 10. Risk Banding and Lending Policy

Translating continuous PD scores into three actionable decision tiers.
Thresholds set at the 30th and 60th percentile, then validated against actual outcomes.


In [ ]:
t1 = float(np.percentile(lr_prob, 30))
t2 = float(np.percentile(lr_prob, 60))

val_df["RISK_BAND"] = val_df["PD_SCORE"].apply(
    lambda s: "Auto Approve" if s < t1 else ("Manual Review" if s < t2 else "Decline")
)

band_order = ["Auto Approve", "Manual Review", "Decline"]
summary = (
    val_df.groupby("RISK_BAND")
    .agg(Count=("PD_SCORE","count"), DefaultRate=("ACTUAL","mean"))
    .loc[band_order]
)
summary["PctPortfolio"] = (summary["Count"] / summary["Count"].sum() * 100).round(1)
summary["DefaultRate"]  = (summary["DefaultRate"] * 100).round(1)

print(f"Thresholds: {t1:.1%} / {t2:.1%}")
print()
print(summary.to_string())

band_colors = [GREEN, AMBER, RED]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(summary["Count"], labels=summary.index, autopct="%1.1f%%",
            colors=band_colors, startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[0].set_title("Portfolio Split by Risk Band", fontweight="bold")

axes[1].bar(summary.index, summary["DefaultRate"], color=band_colors,
            edgecolor="white", width=0.4)
axes[1].set_ylabel("Actual Default Rate (%)")
axes[1].set_title("Actual Default Rate per Band", fontweight="bold")
for i, v in enumerate(summary["DefaultRate"]):
    axes[1].text(i, v + 0.2, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

sep = summary.loc["Decline","DefaultRate"] / summary.loc["Auto Approve","DefaultRate"]
print(f"The Decline band has {sep:.0f}x the default rate of the Auto Approve band.")
print("The thresholds produce meaningfully distinct risk groups.")

### Summary

| Metric | Value |
|---|---|
| AUC-ROC | **0.8895** |
| Gini | **0.7789** |
| KS Statistic | **0.6296** |
| Avg Precision | **0.5977** |

**All case study requirements met:**
- Class imbalance assessed and handled
- Bureau missingness interpreted (thin-file strategy)
- 10 features across 3 MECE dimensions
- Leakage-free pipeline (fit on training data only)
- Baseline model built and evaluated
- High-risk applicant explained via SHAP
- Risk bands validated against actual outcomes
